In [1]:
import os
import json
from dotenv import load_dotenv
from pydantic import (BaseModel, Field, ValidationError)
from typing import (List, Literal)
from azure.identity import (DefaultAzureCredential, get_bearer_token_provider)
from azure.storage.blob import BlobServiceClient
from langchain_openai import (ChatOpenAI, OpenAIEmbeddings)
from langgraph.graph import (StateGraph, START, END)
from langchain.agents import create_agent

In [2]:
load_dotenv()

True

In [3]:
azure_project_endpoint = os.getenv("AZURE_PROJECT_ENDPOINT")
azure_model_endpoint = os.getenv("AZURE_MODEL_ENDPOINT")
azure_model_name = os.getenv("AZURE_MODEL_NAME")
azure_embedding_model_name = os.getenv("AZURE_EMBEDDING_MODEL_NAME")

In [4]:
credential = DefaultAzureCredential()

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)

In [5]:
llm = ChatOpenAI(
    model=azure_model_name,
    base_url=f"{azure_project_endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
    use_responses_api=True,
)

In [6]:
def load_employee_data():
    """Download employee JSON data from Azure Blob Storage."""
    account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
    container_name = os.environ["AZURE_STORAGE_CONTAINER_EMPLOYEES"]
    blob_name = os.environ["AZURE_STORAGE_BLOB_NAME_EMPLOYEES"]

    credential = DefaultAzureCredential()

    try:
        service = BlobServiceClient(
            account_url=account_url,
            credential=credential,
        )

        blob = service.get_blob_client(
            container=container_name,
            blob=blob_name,
        )

        json_bytes = blob.download_blob().readall()
        data = json.loads(json_bytes.decode("utf-8"))

        # Accept either a list or {"employees": [...]}.
        if isinstance(data, dict):
            data = data.get("employees")

        if not isinstance(data, list):
            raise ValueError(
                "Employee data must be a JSON list or an "
                "object containing an 'employees' list."
            )

        return data

    finally:
        credential.close()

In [7]:
def load_task_data():
    """Download task JSON data from Azure Blob Storage."""

    account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
    container_name = os.environ["AZURE_STORAGE_CONTAINER_TASKS"]
    blob_name = os.environ["AZURE_STORAGE_BLOB_NAME_TASKS"]
    
    credential = DefaultAzureCredential()

    try:
        service = BlobServiceClient(
            account_url=account_url,
            credential=credential,
        )

        blob = service.get_blob_client(
            container=container_name,
            blob=blob_name,
        )

        json_bytes = blob.download_blob().readall()
        data = json.loads(json_bytes.decode("utf-8"))

        # Accept either a list or {"tasks": [...]}.
        if isinstance(data, dict):
            data = data.get("tasks")

        if not isinstance(data, list):
            raise ValueError(
                "Task data must be a JSON list or an "
                "object containing a 'tasks' list."
            )

        return data

    finally:
        credential.close()

In [8]:
employee_data = load_employee_data()
employee_data

[{'id': 0,
  'name': 'Jonas Greslund',
  'position': 'Software Engineer',
  'skill_map': {'angular': 'expert',
   'dotnet': 'advanced',
   'python': 'advanced',
   'sql': 'advanced',
   'pulumi': 'novice'}},
 {'id': 1,
  'name': 'Link',
  'position': 'Software Engineer',
  'skill_map': {'angular': 'advanced',
   'dotnet': 'intermediate',
   'python': 'advanced',
   'sql': 'intermediate',
   'pulumi': 'intermediate'}},
 {'id': 2,
  'name': 'Zeenula Pelphegopoulos',
  'position': 'Senior Software Engineer',
  'skill_map': {'angular': 'intermediate',
   'dotnet': 'advanced',
   'python': 'expert',
   'sql': 'advanced',
   'pulumi': 'expert'}},
 {'id': 3,
  'name': 'Peach',
  'position': 'Software Quality Engineer',
  'skill_map': {'selenium': 'advanced', 'regression_testing': 'advanced'}}]

In [9]:
task_data = load_task_data()
task_data

[{'id': 0,
  'title': 'Handle the edge case with the UI of the scoring performance chart.',
  'description': 'There are times when 2 different z-scores have a value of zero.',
  'required_skills': ['Angular']},
 {'id': 1,
  'title': 'Deploy infrastructure',
  'description': 'Provision cloud resources using Pulumi.',
  'required_skills': ['pulumi']},
 {'id': 2,
  'title': 'Optimize database queries',
  'description': 'Improve the performance of database queries to reduce latency.',
  'required_skills': ['sql']},
 {'id': 3,
  'title': 'Conduct regression testing',
  'description': 'Perform regression testing to ensure that new code changes do not break existing functionality.',
  'required_skills': ['regression_testing']}]

In [10]:
def evaluate_employee_skill_set(employee_name, skill, level, evalation_agent):
    """
    This function returns an evaluation of a particular employee's skills.
    """

    evaluation_prompt = f"""
        The employee's name is {employee_name}.
        The skill under evaluation is {skill}.
        The level of expertise is {level}.
    """

    result = evaluation_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ]
    })
    return result

In [11]:
evaluation_agent_system_prompt = (
    "You are an employee skill evaluator. "
    "Your job is to read an employee's skill as well as how advanced they are at that particular skill. "
    "You must output a brief statement about their level at that skill."
)

evaluation_agent = create_agent(
    model=llm,
    tools=[], # We can add tools later.
    system_prompt=evaluation_agent_system_prompt
)

In [12]:
class EmployeeEvaluation(BaseModel):
    employee_id: int
    evaluation: str

In [13]:
evaluations: List[EmployeeEvaluation] = []
for employee in employee_data:
    for skill, level in employee['skill_map'].items():
        print('skill ', skill)
        print('level ', level)
        print('name ', employee['name'])
        evaluation = evaluate_employee_skill_set(employee['name'], skill, level, evaluation_agent)
        evaluation_content = evaluation['messages'][1].content
        evaluation_text = evaluation_content[0]['text']
        print('evaluation_text ', evaluation_text)
        evaluations.append({
            "employee_id": employee["id"],
            "evaluation": evaluation_text
        })

skill  angular
level  expert
name  Jonas Greslund
evaluation_text  Jonas Greslund is an expert in Angular, demonstrating advanced proficiency in building and maintaining complex applications with the framework.
skill  dotnet
level  advanced
name  Jonas Greslund
evaluation_text  Jonas Greslund demonstrates advanced proficiency in .NET, with strong expertise in developing and maintaining complex applications using the framework.
skill  python
level  advanced
name  Jonas Greslund
evaluation_text  Jonas Greslund demonstrates advanced proficiency in Python, with the ability to tackle complex programming tasks and develop robust, efficient solutions.
skill  sql
level  advanced
name  Jonas Greslund
evaluation_text  Jonas Greslund demonstrates advanced proficiency in SQL, with the expertise to handle complex queries and sophisticated database operations.
skill  pulumi
level  novice
name  Jonas Greslund
evaluation_text  Jonas Greslund is a novice in Pulumi, with foundational knowledge and limit

In [14]:
# Initialize with an embedding model
embeddings_model = OpenAIEmbeddings(
    model=azure_embedding_model_name,
    base_url=f"{azure_model_endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)

In [15]:
evaluations

[{'employee_id': 0,
  'evaluation': 'Jonas Greslund is an expert in Angular, demonstrating advanced proficiency in building and maintaining complex applications with the framework.'},
 {'employee_id': 0,
  'evaluation': 'Jonas Greslund demonstrates advanced proficiency in .NET, with strong expertise in developing and maintaining complex applications using the framework.'},
 {'employee_id': 0,
  'evaluation': 'Jonas Greslund demonstrates advanced proficiency in Python, with the ability to tackle complex programming tasks and develop robust, efficient solutions.'},
 {'employee_id': 0,
  'evaluation': 'Jonas Greslund demonstrates advanced proficiency in SQL, with the expertise to handle complex queries and sophisticated database operations.'},
 {'employee_id': 0,
  'evaluation': 'Jonas Greslund is a novice in Pulumi, with foundational knowledge and limited practical experience. He may need guidance to define, deploy, and manage infrastructure using the platform.'},
 {'employee_id': 1,
  '

In [16]:
class TaskEmployeeCandidate(BaseModel):
    learning_opportunity: Literal["Low", "Medium", "High"] = Field(
        description="Employee's learning opportunity. Low means limited opportunity; High means significant opportunity."
    )

    leadership_capability: Literal["Low", "Medium", "High"] = Field(
        description="Employee's technical leadership and mentoring capability."
    )

    learning_explanation: str = Field(
        description="Brief explanation supporting the learning opportunity classification."
    )

    leadership_explanation: str = Field(
        description="Brief explanation supporting the leadership capability classification."
    )

In [17]:
skill_matching_agent_system_prompt = (
    "You are a technical skill evaluation and task-to-employee matching agent. "

    "YOUR OBJECTIVE: "
    "Evaluate an employee's suitability for a task by comparing the "
    "technical skills required by the task against the employee's "
    "existing skills and proficiency levels. Identify the employee's "
    "learning opportunities and potential to provide technical leadership. "

    "SKILL EVALUATION: "
    "Evaluate each required skill against the employee's skill map. "
    "Use demonstrated proficiency as your primary source of evidence. "
    "You may recognize transferable knowledge between related technologies. "
    "For example, Angular experience may suggest familiarity with CSS, "
    "but it does not automatically establish CSS proficiency. "
    "Distinguish demonstrated proficiency from inferred proficiency. "
    "Never invent skills, experience, or qualifications. "

    "LEARNING OPPORTUNITY: "
    "Classify the employee's opportunity to learn and develop the "
    "technical skills required by the task. "
    "LOW: The employee demonstrates advanced or expert proficiency "
    "in most required skills and has limited learning opportunities. "
    "MEDIUM: The employee demonstrates a mixture of proficiency levels "
    "or has some opportunities to improve existing skills. "
    "HIGH: The employee demonstrates beginner-level proficiency in "
    "multiple required skills and has substantial learning opportunities. "

    "LEADERSHIP CAPABILITY: "
    "Classify the employee's potential to provide technical leadership, "
    "mentorship, or guidance to others working on the task. "
    "LOW: The employee demonstrates beginner-level proficiency in most "
    "required skills and has limited demonstrated ability to guide others. "
    "MEDIUM: The employee demonstrates intermediate proficiency or "
    "advanced proficiency in some required skills and may provide "
    "technical guidance in those areas. "
    "HIGH: The employee demonstrates advanced or expert proficiency "
    "in most required skills and has strong potential to mentor others. "

    "DEFAULT CLASSIFICATION RULE: "
    "When in doubt, always default to MEDIUM for both classifications. "
    "If a required skill is missing from the employee's skill map, "
    "treat its proficiency as unknown rather than beginner-level. "
    "Missing information alone is not evidence of a significant learning "
    "opportunity or a lack of leadership capability. "
    "Only assign LOW or HIGH when the available evidence provides "
    "a reasonable basis for doing so. "

    "CLASSIFICATION CONSIDERATIONS: "
    "Evaluate learning opportunity and leadership capability independently. "
    "An employee may have both HIGH learning opportunity and HIGH "
    "leadership capability when they demonstrate expertise in some "
    "required skills but have opportunities to develop in others. "
    "Consider the overall alignment between task requirements and "
    "the employee's skill map. "

    "STRICT OUTPUT REQUIREMENTS: "
    "For learning_opportunity and leadership_capability, return exactly "
    "one of these values: 'Low', 'Medium', or 'High'. "
    "Use the exact capitalization shown. "
    "Never return numerical scores, percentages, or alternative labels. "

    "EXPLANATION REQUIREMENTS: "
    "Provide a brief, evidence-based explanation for each classification. "
    "Reference specific required skills and the employee's demonstrated "
    "proficiency where applicable. "
    "Do not make unsupported assumptions about the employee's abilities. "

    "Your final response must conform to the required structured output schema."
)

skill_matching_agent = create_agent(
    model=llm,
    tools=[], # We can add tools later.
    system_prompt=evaluation_agent_system_prompt,
    response_format=TaskEmployeeCandidate
)

In [ ]:
def match_task_to_skills(task_id, skills, employee, skill_matching_agent) -> (TaskEmployeeCandidate, int, int):
    print('task_id ', task_id)
    print('skills ', skills)
    print('employee ', employee)

    skill_matching_prompt = f"""
        The required skills are: {skills}
        The employee's skills are: {employee['skill_map']}
    """
    print('skill_matching_prompt ', skill_matching_prompt)

    try:
        result = skill_matching_agent.invoke({
            "messages": [
                {"role": "user", "content": skill_matching_prompt}
            ]
        })
        print('structured response ', result["structured_response"])

        return (result["structured_response"], task_id, employee['id'])

    except (ValidationError, Exception) as e:
        print(f"Skill matching agent failed: {e}")

        return (TaskEmployeeCandidate(
            learning_opportunity="Medium",
            leadership_capability="Medium",
            learning_explanation="Unable to determine learning opportunity.",
            leadership_explanation="Unable to determine leadership capability."
        ), task_id, employee['id'])

In [19]:
employee_data

[{'id': 0,
  'name': 'Jonas Greslund',
  'position': 'Software Engineer',
  'skill_map': {'angular': 'expert',
   'dotnet': 'advanced',
   'python': 'advanced',
   'sql': 'advanced',
   'pulumi': 'novice'}},
 {'id': 1,
  'name': 'Link',
  'position': 'Software Engineer',
  'skill_map': {'angular': 'advanced',
   'dotnet': 'intermediate',
   'python': 'advanced',
   'sql': 'intermediate',
   'pulumi': 'intermediate'}},
 {'id': 2,
  'name': 'Zeenula Pelphegopoulos',
  'position': 'Senior Software Engineer',
  'skill_map': {'angular': 'intermediate',
   'dotnet': 'advanced',
   'python': 'expert',
   'sql': 'advanced',
   'pulumi': 'expert'}},
 {'id': 3,
  'name': 'Peach',
  'position': 'Software Quality Engineer',
  'skill_map': {'selenium': 'advanced', 'regression_testing': 'advanced'}}]

In [20]:
skill_matches = []
for task in task_data:
    skills = ", ".join(task['required_skills'])
    for employee in employee_data:
        skill_match = match_task_to_skills(task['id'], skills, employee, skill_matching_agent)
        skill_matches.append(skill_match)

task_id  0
skills  Angular
employee  {'id': 0, 'name': 'Jonas Greslund', 'position': 'Software Engineer', 'skill_map': {'angular': 'expert', 'dotnet': 'advanced', 'python': 'advanced', 'sql': 'advanced', 'pulumi': 'novice'}}
skill_matching_prompt  
        The required skills are: Angular
        The employee's skills are: {'angular': 'expert', 'dotnet': 'advanced', 'python': 'advanced', 'sql': 'advanced', 'pulumi': 'novice'}
    
structured response  learning_opportunity='Low' leadership_capability='High' learning_explanation='The employee is an expert in Angular, so the required skill offers limited foundational learning opportunity.' leadership_explanation='Expert-level Angular proficiency suggests strong potential for technical guidance and mentoring, though actual leadership experience is not provided.'
task_id  0
skills  Angular
employee  {'id': 1, 'name': 'Link', 'position': 'Software Engineer', 'skill_map': {'angular': 'advanced', 'dotnet': 'intermediate', 'python': 'advanced',

In [21]:
for skill_match in skill_matches:
    print(skill_match)
    print('-')

(TaskEmployeeCandidate(learning_opportunity='Low', leadership_capability='High', learning_explanation='The employee is an expert in Angular, so the required skill offers limited foundational learning opportunity.', leadership_explanation='Expert-level Angular proficiency suggests strong potential for technical guidance and mentoring, though actual leadership experience is not provided.'), 0, 0)
-
(TaskEmployeeCandidate(learning_opportunity='Low', leadership_capability='High', learning_explanation='The employee has advanced Angular skills, so the required skill offers limited foundational learning opportunity.', leadership_explanation='Advanced Angular expertise suggests strong potential for technical guidance and mentoring, though leadership experience is not specified.'), 0, 1)
-
(TaskEmployeeCandidate(learning_opportunity='Medium', leadership_capability='Medium', learning_explanation='The employee has intermediate Angular skills, with room to develop advanced framework expertise.', l

In [22]:
SCORE_MAP = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

for skill_match in skill_matches:
    learning_score = SCORE_MAP[skill_match[0].learning_opportunity]
    leadership_score = SCORE_MAP[skill_match[0].leadership_capability]
    print('learning_score ', learning_score)
    print('leadership_score ', leadership_score)

learning_score  0
leadership_score  2
learning_score  0
leadership_score  2
learning_score  1
leadership_score  1
learning_score  2
leadership_score  0
learning_score  2
leadership_score  0
learning_score  1
leadership_score  1
learning_score  0
leadership_score  2
learning_score  2
leadership_score  0
learning_score  1
leadership_score  1
learning_score  1
leadership_score  1
learning_score  0
leadership_score  2
learning_score  2
leadership_score  0
learning_score  2
leadership_score  0
learning_score  2
leadership_score  0
learning_score  2
leadership_score  0
learning_score  0
leadership_score  2


In [23]:
learning_ranked = sorted(
    skill_matches,
    key=lambda x: SCORE_MAP[x[0].learning_opportunity]
)
learning_ranked

[(TaskEmployeeCandidate(learning_opportunity='Low', leadership_capability='High', learning_explanation='The employee is an expert in Angular, so the required skill offers limited foundational learning opportunity.', leadership_explanation='Expert-level Angular proficiency suggests strong potential for technical guidance and mentoring, though actual leadership experience is not provided.'),
  0,
  0),
 (TaskEmployeeCandidate(learning_opportunity='Low', leadership_capability='High', learning_explanation='The employee has advanced Angular skills, so the required skill offers limited foundational learning opportunity.', leadership_explanation='Advanced Angular expertise suggests strong potential for technical guidance and mentoring, though leadership experience is not specified.'),
  0,
  1),
 (TaskEmployeeCandidate(learning_opportunity='Low', leadership_capability='High', learning_explanation='The employee is an expert in Pulumi, so there is limited opportunity to learn the required skill

In [24]:
leadership_ranked = sorted(
    skill_matches,
    key=lambda x: SCORE_MAP[x[0].leadership_capability],
)
leadership_ranked

[(TaskEmployeeCandidate(learning_opportunity='High', leadership_capability='Low', learning_explanation='Angular proficiency is not demonstrated. Advanced Selenium and regression-testing skills provide a useful testing foundation, but Angular development offers significant learning opportunity.', leadership_explanation='Angular-specific technical leadership and mentoring capability is not demonstrated by the listed testing skills.'),
  0,
  3),
 (TaskEmployeeCandidate(learning_opportunity='High', leadership_capability='Low', learning_explanation='The employee is a novice in Pulumi, indicating significant opportunity to develop proficiency.', leadership_explanation='Their novice Pulumi proficiency suggests limited readiness to provide technical leadership or mentor others in this skill.'),
  1,
  0),
 (TaskEmployeeCandidate(learning_opportunity='High', leadership_capability='Low', learning_explanation="Pulumi is not listed among the employee's skills, indicating a significant opportunity

In [25]:
print(len(leadership_ranked))
print(len(learning_ranked))

16
16
